In [1]:
# Import libraries and define configurations
import json
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import joblib
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

RANDOM_STATE = 121
N_JOBS = -1

np.random.seed(RANDOM_STATE)


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_WORKING_DIRECTORY.parent
    if CURRENT_WORKING_DIRECTORY.name == "notebooks"
    else CURRENT_WORKING_DIRECTORY
)

PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

ENGINEERED_FEATURES_PARQUET_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.parquet"
)
ENGINEERED_FEATURES_CSV_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.csv"
)
CLEANED_DONOR_DATA_PATH = (
    PROCESSED_DATA_DIRECTORY / "cleaned_donor_data.csv"
)
FEATURE_DICTIONARY_PATH = (
    REPORTS_DIRECTORY / "feature_dictionary.csv"
)

MODEL_PREDICTIONS_PATH = (
    PROCESSED_DATA_DIRECTORY / "model_predictions.csv"
)
FINAL_PRIMARY_PIPELINE_PATH = (
    MODELS_DIRECTORY / "final_primary_donor_pipeline.joblib"
)
MODEL_COMPARISON_RESULTS_PATH = (
    REPORTS_DIRECTORY / "05_model_comparison_results.csv"
)
CLASSIFICATION_MODELING_REPORT_PATH = (
    REPORTS_DIRECTORY / "05_classification_modeling.md"
)

MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)
REPORTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")
set_config(display="diagram")

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Load the engineered dataset and feature dictionary
def format_project_path(path):
    relative_path = Path(path).resolve().relative_to(PROJECT_ROOT)
    return f"/{PROJECT_ROOT.name}/{relative_path.as_posix()}"


artifact_availability = pd.DataFrame({
    "artifact": [
        "Engineered features Parquet",
        "Engineered features CSV fallback",
        "Feature dictionary",
        "Cleaned donor dataset",
    ],
    "path_object": [
        ENGINEERED_FEATURES_PARQUET_PATH,
        ENGINEERED_FEATURES_CSV_PATH,
        FEATURE_DICTIONARY_PATH,
        CLEANED_DONOR_DATA_PATH,
    ],
})

artifact_availability["path"] = artifact_availability["path_object"].apply(
    format_project_path
)
artifact_availability["available"] = artifact_availability["path_object"].apply(
    Path.exists
)

display(artifact_availability[
    ["artifact", "path", "available"]
].style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
)

if ENGINEERED_FEATURES_PARQUET_PATH.exists():
    modeling_data = pd.read_parquet(
        ENGINEERED_FEATURES_PARQUET_PATH
    )
    modeling_data_source = "Parquet"
    modeling_data_path = ENGINEERED_FEATURES_PARQUET_PATH

elif ENGINEERED_FEATURES_CSV_PATH.exists():
    modeling_data = pd.read_csv(
        ENGINEERED_FEATURES_CSV_PATH
    )
    modeling_data_source = "CSV fallback"
    modeling_data_path = ENGINEERED_FEATURES_CSV_PATH

else:
    raise FileNotFoundError(
        "Neither donor_features.parquet nor donor_features.csv "
        "was found in the processed data directory."
    )

if not FEATURE_DICTIONARY_PATH.exists():
    raise FileNotFoundError(
        f"Feature dictionary not found: "
        f"{format_project_path(FEATURE_DICTIONARY_PATH)}"
    )

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_PATH
)

print(f"\nModeling dataset source: {modeling_data_source}")
print(
    "Modeling dataset path:",
    format_project_path(modeling_data_path),
)
print(
    "Feature dictionary path:",
    format_project_path(FEATURE_DICTIONARY_PATH),
)

print(
    "\nModeling dataset loaded:",
    f"{modeling_data.shape[0]:,} rows × "
    f"{modeling_data.shape[1]:,} columns",
)

print(
    "Feature dictionary loaded:",
    f"{feature_dictionary.shape[0]:,} rows × "
    f"{feature_dictionary.shape[1]:,} columns\n",
)

artifact,path,available
Engineered features Parquet,/red-cross-donor-prediction/data/processed/donor_features.parquet,True
Engineered features CSV fallback,/red-cross-donor-prediction/data/processed/donor_features.csv,True
Feature dictionary,/red-cross-donor-prediction/reports/feature_dictionary.csv,True
Cleaned donor dataset,/red-cross-donor-prediction/data/processed/cleaned_donor_data.csv,True



Modeling dataset source: Parquet
Modeling dataset path: /red-cross-donor-prediction/data/processed/donor_features.parquet
Feature dictionary path: /red-cross-donor-prediction/reports/feature_dictionary.csv

Modeling dataset loaded: 34,403 rows × 55 columns
Feature dictionary loaded: 77 rows × 7 columns



In [3]:
# Validate the Phase 4 modeling export
EXPECTED_ROW_COUNT = 34_403
EXPECTED_COLUMN_COUNT = 55
EXPECTED_PREDICTOR_COUNT = 53

TRACKING_IDENTIFIER_COLUMN = "donor_unique_id"
PRIMARY_TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

DIRECT_LEAKAGE_COLUMNS = {
    "current_fiscal_year_donation",
    "cumulative_donation_amount",
}

EXPECTED_TARGET_COUNTS = {
    0: 32_499,
    1: 1_904,
}

excluded_modeling_columns = {
    TRACKING_IDENTIFIER_COLUMN,
    PRIMARY_TARGET_COLUMN,
}

predictor_columns = [
    column
    for column in modeling_data.columns
    if column not in excluded_modeling_columns
]

unexpected_direct_leakage_columns = sorted(
    set(predictor_columns).intersection(DIRECT_LEAKAGE_COLUMNS)
)

numeric_columns = modeling_data.select_dtypes(
    include=[np.number]
).columns

infinite_value_count = int(
    np.isinf(modeling_data[numeric_columns]).sum().sum()
)

target_distribution = (
    modeling_data[PRIMARY_TARGET_COLUMN]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("target_class")
    .reset_index(name="record_count")
)

target_distribution["percentage"] = (
    target_distribution["record_count"]
    / len(modeling_data)
    * 100
)

target_distribution["expected_record_count"] = (
    target_distribution["target_class"]
    .map(EXPECTED_TARGET_COUNTS)
)

target_distribution["matches_expected"] = (
    target_distribution["record_count"]
    == target_distribution["expected_record_count"]
)

actual_target_counts = target_distribution.set_index(
    "target_class"
)["record_count"].to_dict()

validation_results = pd.DataFrame({
    "validation_check": [
        "Record count",
        "Total column count",
        "Predictor count",
        "Tracking identifier count",
        "Primary target count",
        "Missing tracking identifiers",
        "Duplicate tracking identifiers",
        "Unexpected direct-leakage columns",
        "Infinite numeric values",
        "Primary target class 0 count",
        "Primary target class 1 count",
    ],
    "expected": [
        EXPECTED_ROW_COUNT,
        EXPECTED_COLUMN_COUNT,
        EXPECTED_PREDICTOR_COUNT,
        1,
        1,
        0,
        0,
        "None",
        0,
        EXPECTED_TARGET_COUNTS[0],
        EXPECTED_TARGET_COUNTS[1],
    ],
    "actual": [
        modeling_data.shape[0],
        modeling_data.shape[1],
        len(predictor_columns),
        list(modeling_data.columns).count(
            TRACKING_IDENTIFIER_COLUMN
        ),
        list(modeling_data.columns).count(
            PRIMARY_TARGET_COLUMN
        ),
        modeling_data[
            TRACKING_IDENTIFIER_COLUMN
        ].isna().sum(),
        modeling_data[
            TRACKING_IDENTIFIER_COLUMN
        ].duplicated().sum(),
        (
            ", ".join(unexpected_direct_leakage_columns)
            if unexpected_direct_leakage_columns
            else "None"
        ),
        infinite_value_count,
        actual_target_counts[0],
        actual_target_counts[1],
    ],
})

validation_results["passed"] = [
    modeling_data.shape[0] == EXPECTED_ROW_COUNT,
    modeling_data.shape[1] == EXPECTED_COLUMN_COUNT,
    len(predictor_columns) == EXPECTED_PREDICTOR_COUNT,
    list(modeling_data.columns).count(
        TRACKING_IDENTIFIER_COLUMN
    ) == 1,
    list(modeling_data.columns).count(
        PRIMARY_TARGET_COLUMN
    ) == 1,
    modeling_data[
        TRACKING_IDENTIFIER_COLUMN
    ].isna().sum() == 0,
    modeling_data[
        TRACKING_IDENTIFIER_COLUMN
    ].duplicated().sum() == 0,
    len(unexpected_direct_leakage_columns) == 0,
    infinite_value_count == 0,
    actual_target_counts[0] == EXPECTED_TARGET_COUNTS[0],
    actual_target_counts[1] == EXPECTED_TARGET_COUNTS[1],
]

def format_validation_value(value):
    if isinstance(value, (bool, np.bool_)):
        return str(value)

    if isinstance(value, (int, np.integer)):
        return f"{value:,.0f}"

    if isinstance(value, (float, np.floating)):
        if pd.isna(value):
            return ""

        if value.is_integer():
            return f"{value:,.0f}"

        if value != 0 and abs(value) < 0.01:
            return f"{value:,.6f}".rstrip("0").rstrip(".")

        return f"{value:,.2f}"

    return str(value)


display(validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(target_distribution.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
        "expected_record_count": "{:,.0f}",
    })
)

failed_validation_checks = validation_results.loc[
    ~validation_results["passed"],
    "validation_check",
].tolist()

if failed_validation_checks:
    raise AssertionError(
        "Phase 4 modeling export validation failed for: "
        + ", ".join(failed_validation_checks)
    )

print("\nAll Phase 4 modeling export validation checks passed.\n")

validation_check,expected,actual,passed
Record count,"34,403","34,403",True
Total column count,55,55,True
Predictor count,53,53,True
Tracking identifier count,1,1,True
Primary target count,1,1,True
Missing tracking identifiers,0,0,True
Duplicate tracking identifiers,0,0,True
Unexpected direct-leakage columns,None,None,True
Infinite numeric values,0,0,True
Primary target class 0 count,"32,499","32,499",True


target_class,record_count,percentage,expected_record_count,matches_expected
0,"32,499",94.47%,"32,499",True
1,"1,904",5.53%,"1,904",True



All Phase 4 modeling export validation checks passed.



## Modeling Setup and Data Validation

The Phase 5 modeling environment was configured using a consistent random state of `121`, reusable project paths, and the libraries required for preprocessing, classification, model evaluation, visualization, and model persistence.

Project paths are defined relative to the repository root so the notebook does not display user-specific local directories.

The engineered modeling dataset was loaded from the Parquet export, with the CSV retained as a fallback. The feature dictionary and original cleaned dataset were also confirmed to be available.

The modeling export contains:

* 34,403 records
* 55 total columns
* 53 leakage-safe predictors
* 1 tracking identifier
* 1 primary target

The `donor_unique_id` field contains no missing or duplicate values. No direct-leakage columns or infinite numeric values were found.

The primary target distribution remains unchanged:

| Target Class | Records | Percentage |
| ------------ | ------: | ---------: |
| 0            |  32,499 |     94.47% |
| 1            |   1,904 |      5.53% |

All Phase 4 modeling export validation checks passed. The severe class imbalance confirms that Phase 5 should use stratified splitting and evaluation metrics beyond accuracy.
